# Pipeline ACEU - Notebook Exploratório

Este notebook permite validar visualmente cada componente do modelo ACEU
antes de consolidar os resultados finais.

**Modelo:** Hectares Indicator (ECOMETRICA, 2019)  
**Área:** Mato Grosso  
**Resolução:** 30 metros (EPSG:31981)

---

In [ ]:
# Imports
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
import rasterio
from rasterio.plot import show
import geopandas as gpd

# Configurações de visualização
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Diretórios
from config import RASTERS_DIR, DADOS_BRUTOS_DIR, OUTPUT_DIR

print(f"Diretório de rasters: {RASTERS_DIR}")
print(f"Arquivos disponíveis: {os.listdir(RASTERS_DIR) if os.path.exists(RASTERS_DIR) else 'vazio'}")

## 1. Grade de Referência e Máscara do MT

Verificar se a grade foi criada corretamente e se a máscara do estado
cobre a área esperada (~903.000 km²).

In [ ]:
# Carregar máscara do MT
with rasterio.open(os.path.join(RASTERS_DIR, 'mascara_mt.tif')) as src:
    mascara = src.read(1)
    meta = src.meta
    transform = src.transform
    bounds = src.bounds

print(f"Dimensões: {mascara.shape[1]} x {mascara.shape[0]} pixels")
print(f"Resolução: {transform.a:.1f} m")
print(f"CRS: {meta['crs']}")
print(f"Bounds: {bounds}")

pixels_dentro = np.sum(mascara == 1)
area_km2 = pixels_dentro * (30**2) / 1e6
print(f"\nPixels dentro do MT: {pixels_dentro:,}")
print(f"Área estimada: {area_km2:,.0f} km² (referência IBGE: 903.357 km²)")

# Visualizar
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
ax.imshow(mascara, cmap='Greens', interpolation='nearest')
ax.set_title('Máscara do Mato Grosso (1 = dentro, 0 = fora)')
ax.set_xlabel('Coluna (pixel)')
ax.set_ylabel('Linha (pixel)')
plt.tight_layout()
plt.show()

## 2. Componente A — Acessibilidade

Distância euclidiana às rodovias, reclassificada em 5 classes.  
Classe 5 = muito acessível (perto de rodovia) = maior pressão.

In [ ]:
# Carregar componente A
caminho_a = os.path.join(RASTERS_DIR, 'componente_a.tif')
if os.path.exists(caminho_a):
    with rasterio.open(caminho_a) as src:
        comp_a = src.read(1)
    
    # Estatísticas
    print("Distribuição do Componente A (Acessibilidade):")
    for classe in range(1, 6):
        n = np.sum(comp_a == classe)
        pct = n / np.sum(mascara == 1) * 100
        print(f"  Classe {classe}: {n:>12,} pixels ({pct:5.1f}%)")
    
    # Visualizar
    cmap_aceu = ListedColormap(['#228B22', '#90EE90', '#FFFF00', '#FFA500', '#DC1414'])
    bounds_plot = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
    norm = BoundaryNorm(bounds_plot, cmap_aceu.N)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Raster de distância
    caminho_dist = os.path.join(RASTERS_DIR, 'distancia_rodovias_mt.tif')
    if os.path.exists(caminho_dist):
        with rasterio.open(caminho_dist) as src:
            dist = src.read(1)
        dist_masked = np.ma.masked_where(mascara == 0, dist)
        im0 = axes[0].imshow(dist_masked / 1000, cmap='YlOrRd_r', interpolation='nearest')
        axes[0].set_title('Distância a rodovias (km)')
        plt.colorbar(im0, ax=axes[0], shrink=0.7, label='km')
    
    # Classes de acessibilidade
    comp_a_masked = np.ma.masked_where(comp_a == 0, comp_a)
    im1 = axes[1].imshow(comp_a_masked, cmap=cmap_aceu, norm=norm, interpolation='nearest')
    axes[1].set_title('Componente A - Classes de Acessibilidade')
    
    # Legenda
    labels = ['1 - Muito baixa', '2 - Baixa', '3 - Média', '4 - Alta', '5 - Muito alta']
    patches = [mpatches.Patch(color=cmap_aceu(i), label=labels[i]) for i in range(5)]
    axes[1].legend(handles=patches, loc='lower right', fontsize=8)
    
    plt.tight_layout()
    plt.show()
else:
    print(f"Arquivo não encontrado: {caminho_a}")
    print("Execute: python 03_acessibilidade.py")

## 3. Componente U — Proteção

Áreas legalmente protegidas (UCs, TIs, Quilombos).  
Binário: 1 = protegido, 0 = não protegido.

In [ ]:
# Carregar componente U
caminho_u = os.path.join(RASTERS_DIR, 'componente_u.tif')
if os.path.exists(caminho_u):
    with rasterio.open(caminho_u) as src:
        comp_u = src.read(1)
    
    pixels_protegidos = np.sum(comp_u == 1)
    pct_protegido = pixels_protegidos / np.sum(mascara == 1) * 100
    area_prot_km2 = pixels_protegidos * (30**2) / 1e6
    
    print(f"Pixels protegidos: {pixels_protegidos:,}")
    print(f"Área protegida: {area_prot_km2:,.0f} km² ({pct_protegido:.1f}% do MT)")
    
    # Visualizar
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    cmap_u = ListedColormap(['#f0f0f0', '#2E7D32'])
    comp_u_masked = np.ma.masked_where(mascara == 0, comp_u)
    ax.imshow(comp_u_masked, cmap=cmap_u, interpolation='nearest')
    ax.set_title(f'Componente U - Áreas Protegidas ({pct_protegido:.1f}% do MT)')
    patches = [
        mpatches.Patch(color='#f0f0f0', label='Não protegido'),
        mpatches.Patch(color='#2E7D32', label='Protegido (UC/TI/Quilombo)'),
    ]
    ax.legend(handles=patches, loc='lower right')
    plt.tight_layout()
    plt.show()
else:
    print(f"Arquivo não encontrado: {caminho_u}")
    print("Execute: python 04_protecao.py")

## 4. Componente C — Cultivabilidade

Proporção de uso agropecuário na vizinhança (proxy MapBiomas).  
Classe 5 = altíssima pressão de conversão (>70% agro ao redor).

In [ ]:
# Carregar componente C
caminho_c = os.path.join(RASTERS_DIR, 'componente_c.tif')
if os.path.exists(caminho_c):
    with rasterio.open(caminho_c) as src:
        comp_c = src.read(1)
    
    print("Distribuição do Componente C (Cultivabilidade):")
    for classe in range(1, 6):
        n = np.sum(comp_c == classe)
        pct = n / np.sum(mascara == 1) * 100
        print(f"  Classe {classe}: {n:>12,} pixels ({pct:5.1f}%)")
    
    # Visualizar
    cmap_c = ListedColormap(['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c'])
    bounds_plot = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
    norm = BoundaryNorm(bounds_plot, cmap_c.N)
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    comp_c_masked = np.ma.masked_where(comp_c == 0, comp_c)
    ax.imshow(comp_c_masked, cmap=cmap_c, norm=norm, interpolation='nearest')
    ax.set_title('Componente C - Cultivabilidade (pressão agropecuária)')
    labels = ['1 - Muito baixa (<10%)', '2 - Baixa (10-30%)', '3 - Média (30-50%)', 
              '4 - Alta (50-70%)', '5 - Muito alta (>70%)']
    patches = [mpatches.Patch(color=cmap_c(i), label=labels[i]) for i in range(5)]
    ax.legend(handles=patches, loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print(f"Arquivo não encontrado: {caminho_c}")
    print("Execute: python 05_cultivabilidade.py")

## 5. Componente E — Extraibilidade

Combinação de recurso florestal (MapBiomas) e pressão mineral (SIGMINE).  
Classe 5 = máximo recurso disponível para extração.

In [ ]:
# Carregar componente E
caminho_e = os.path.join(RASTERS_DIR, 'componente_e.tif')
if os.path.exists(caminho_e):
    with rasterio.open(caminho_e) as src:
        comp_e = src.read(1)
    
    print("Distribuição do Componente E (Extraibilidade):")
    for classe in range(1, 6):
        n = np.sum(comp_e == classe)
        pct = n / np.sum(mascara == 1) * 100
        print(f"  Classe {classe}: {n:>12,} pixels ({pct:5.1f}%)")
    
    # Visualizar
    cmap_e = ListedColormap(['#eff3ff', '#bdd7e7', '#6baed6', '#3182bd', '#08519c'])
    bounds_plot = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
    norm = BoundaryNorm(bounds_plot, cmap_e.N)
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    comp_e_masked = np.ma.masked_where(comp_e == 0, comp_e)
    ax.imshow(comp_e_masked, cmap=cmap_e, norm=norm, interpolation='nearest')
    ax.set_title('Componente E - Extraibilidade (recurso florestal + mineral)')
    labels = ['1 - Sem recurso', '2 - Baixo', '3 - Médio', '4 - Alto', '5 - Muito alto']
    patches = [mpatches.Patch(color=cmap_e(i), label=labels[i]) for i in range(5)]
    ax.legend(handles=patches, loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print(f"Arquivo não encontrado: {caminho_e}")
    print("Execute: python 06_extraibilidade.py")

## 6. Risco ACEU Final

Composição: R(x) = A(x) + C(x) + E(x) - U(x)  
Classificado em quintis sobre pixels de floresta de referência.

In [ ]:
# Carregar risco ACEU
caminho_aceu = os.path.join(RASTERS_DIR, 'risco_aceu.tif')
if os.path.exists(caminho_aceu):
    with rasterio.open(caminho_aceu) as src:
        risco = src.read(1)
    
    print("Distribuição das Classes de Risco ACEU:")
    total_floresta = np.sum(risco > 0)
    for classe in range(1, 6):
        n = np.sum(risco == classe)
        pct = n / total_floresta * 100 if total_floresta > 0 else 0
        area_km2 = n * (30**2) / 1e6
        print(f"  Classe {classe}: {n:>12,} pixels ({pct:5.1f}%) = {area_km2:,.0f} km²")
    
    # Visualizar
    cmap_risco = ListedColormap(['#228B22', '#90EE90', '#FFFF00', '#FFA500', '#DC1414'])
    bounds_plot = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
    norm = BoundaryNorm(bounds_plot, cmap_risco.N)
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 9))
    risco_masked = np.ma.masked_where(risco == 0, risco)
    ax.imshow(risco_masked, cmap=cmap_risco, norm=norm, interpolation='nearest')
    ax.set_title('RISCO ACEU - Probabilidade de Desmatamento (Mato Grosso)', fontsize=14)
    
    labels = [
        '1 - Risco muito baixo (prob. 10%)',
        '2 - Risco baixo (prob. 30%)',
        '3 - Risco médio (prob. 50%)',
        '4 - Risco alto (prob. 70%)',
        '5 - Risco muito alto (prob. 90%)'
    ]
    patches = [mpatches.Patch(color=cmap_risco(i), label=labels[i]) for i in range(5)]
    ax.legend(handles=patches, loc='lower right', fontsize=9, framealpha=0.9)
    ax.set_xlabel('Coluna (pixel)')
    ax.set_ylabel('Linha (pixel)')
    plt.tight_layout()
    plt.show()
else:
    print(f"Arquivo não encontrado: {caminho_aceu}")
    print("Execute: python 07_composicao_aceu.py")

## 7. Comparação dos 4 Componentes

Visualização lado a lado para identificar padrões espaciais.

In [ ]:
# Painel comparativo
componentes = {
    'A - Acessibilidade': ('componente_a.tif', 'YlOrRd'),
    'U - Proteção': ('componente_u.tif', 'Greens'),
    'C - Cultivabilidade': ('componente_c.tif', 'YlOrRd'),
    'E - Extraibilidade': ('componente_e.tif', 'Blues'),
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, (titulo, (arquivo, cmap_nome)) in enumerate(componentes.items()):
    caminho = os.path.join(RASTERS_DIR, arquivo)
    if os.path.exists(caminho):
        with rasterio.open(caminho) as src:
            dados = src.read(1)
        dados_masked = np.ma.masked_where(dados == 0, dados)
        axes[i].imshow(dados_masked, cmap=cmap_nome, interpolation='nearest')
        axes[i].set_title(titulo, fontsize=12)
    else:
        axes[i].text(0.5, 0.5, f'Não disponível\n{arquivo}', 
                     ha='center', va='center', transform=axes[i].transAxes)
        axes[i].set_title(titulo, fontsize=12, color='gray')
    axes[i].set_xticks([])
    axes[i].set_yticks([])

plt.suptitle('Componentes ACEU - Mato Grosso', fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

## 8. Estatísticas por Município

Análise dos resultados do CSV de desmatamento evitado.

In [ ]:
import pandas as pd

# Carregar CSV de áreas por classe
caminho_csv = os.path.join(OUTPUT_DIR, 'areas_risco_municipios.csv')
if os.path.exists(caminho_csv):
    df = pd.read_csv(caminho_csv)
    print(f"Municípios processados: {len(df)}")
    print(f"\nResumo:")
    print(f"  Área florestal total: {df['area_floresta_total_ha'].sum():,.0f} ha")
    print(f"  Perda esperada em 20 anos: {df['perda_esperada_20anos_km2'].sum():,.0f} km²")
    
    # Top 10 municípios com maior perda esperada
    print(f"\nTop 10 municípios - maior perda esperada (20 anos):")
    top10 = df.nlargest(10, 'perda_esperada_20anos_km2')
    for _, row in top10.iterrows():
        print(f"  {row['cod_municipio']}: {row['perda_esperada_20anos_km2']:,.1f} km²")
    
    # Gráfico de barras
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    top20 = df.nlargest(20, 'perda_esperada_20anos_km2')
    ax.barh(range(len(top20)), top20['perda_esperada_20anos_km2'], color='#d62728')
    ax.set_yticks(range(len(top20)))
    ax.set_yticklabels(top20['cod_municipio'].astype(str), fontsize=8)
    ax.set_xlabel('Perda esperada em 20 anos (km²)')
    ax.set_title('Top 20 Municípios - Maior Perda Florestal Esperada')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print(f"CSV não encontrado: {caminho_csv}")
    print("Execute: python 08_estatisticas.py")

## 9. Validação: Histograma do Risco Bruto

Verificar a distribuição do risco bruto antes da classificação em quintis.

In [ ]:
# Histograma do risco bruto
caminho_bruto = os.path.join(RASTERS_DIR, 'risco_bruto.tif')
if os.path.exists(caminho_bruto):
    with rasterio.open(caminho_bruto) as src:
        risco_bruto = src.read(1)
    
    # Apenas pixels dentro do MT com valor > 0
    valores = risco_bruto[(mascara == 1) & (risco_bruto > 0)]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histograma
    axes[0].hist(valores, bins=50, color='#2E7D32', alpha=0.7, edgecolor='white')
    axes[0].set_xlabel('Risco bruto (A + C + E - U)')
    axes[0].set_ylabel('Frequência (pixels)')
    axes[0].set_title('Distribuição do Risco Bruto')
    axes[0].axvline(np.median(valores), color='red', linestyle='--', label=f'Mediana: {np.median(valores):.1f}')
    axes[0].axvline(np.mean(valores), color='blue', linestyle='--', label=f'Média: {np.mean(valores):.1f}')
    axes[0].legend()
    
    # Boxplot
    axes[1].boxplot(valores, vert=True)
    axes[1].set_ylabel('Risco bruto')
    axes[1].set_title('Boxplot do Risco Bruto')
    
    # Quintis
    percentis = np.percentile(valores, [20, 40, 60, 80])
    for p, val in zip([20, 40, 60, 80], percentis):
        axes[0].axvline(val, color='gray', linestyle=':', alpha=0.5)
    
    print(f"Estatísticas do risco bruto:")
    print(f"  Min: {valores.min()}, Max: {valores.max()}")
    print(f"  Média: {valores.mean():.2f}, Mediana: {np.median(valores):.1f}")
    print(f"  Quintis: P20={percentis[0]:.1f}, P40={percentis[1]:.1f}, P60={percentis[2]:.1f}, P80={percentis[3]:.1f}")
    
    plt.tight_layout()
    plt.show()
else:
    print(f"Arquivo não encontrado: {caminho_bruto}")
    print("Execute: python 07_composicao_aceu.py")

## 10. Zoom em Região de Interesse

Visualizar uma região específica em detalhe para verificar a resolução.

In [ ]:
# Zoom numa região (ajuste as coordenadas conforme interesse)
# Exemplo: região central do MT (Alta Floresta / Sinop)
if os.path.exists(caminho_aceu):
    with rasterio.open(caminho_aceu) as src:
        risco = src.read(1)
    
    # Recorte (ajuste conforme necessário)
    h, w = risco.shape
    # Pegar uma janela de ~100km x 100km no centro-norte
    row_start = int(h * 0.2)
    row_end = int(h * 0.4)
    col_start = int(w * 0.3)
    col_end = int(w * 0.6)
    
    zoom = risco[row_start:row_end, col_start:col_end]
    
    cmap_risco = ListedColormap(['#228B22', '#90EE90', '#FFFF00', '#FFA500', '#DC1414'])
    bounds_plot = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
    norm = BoundaryNorm(bounds_plot, cmap_risco.N)
    
    fig, ax = plt.subplots(1, 1, figsize=(14, 8))
    zoom_masked = np.ma.masked_where(zoom == 0, zoom)
    ax.imshow(zoom_masked, cmap=cmap_risco, norm=norm, interpolation='nearest')
    ax.set_title('Zoom - Risco ACEU (região centro-norte do MT)', fontsize=13)
    
    labels = ['1 - Muito baixo', '2 - Baixo', '3 - Médio', '4 - Alto', '5 - Muito alto']
    patches = [mpatches.Patch(color=cmap_risco(i), label=labels[i]) for i in range(5)]
    ax.legend(handles=patches, loc='lower right', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Janela: linhas {row_start}-{row_end}, colunas {col_start}-{col_end}")
    print(f"Tamanho: {(row_end-row_start)*30/1000:.0f} km x {(col_end-col_start)*30/1000:.0f} km")

---

## Referências

- ECOMETRICA. The Hectares Indicator Methods and Guidance. Version 2.0. Edinburgh, 2019.
- VENDRUSCULO, L.G. et al. Aplicação da metodologia do Hectare Indicator para estimativa de desmatamento evitado no bioma Amazônia. Embrapa Informática Agropecuária, 2019.
- MapBiomas. Collection 9 - Annual Land Use and Land Cover Maps of Brazil. 2024.
- INPE/PRODES. Monitoramento do Desmatamento da Floresta Amazônica Brasileira por Satélite. TerraBrasilis, 2024.